# Chapter 8. Psychometric functions

**Learning objective.** Fit and audit trial-level response functions while separating observed classification behavior from claims about categorical perception.

**Estimated time:** 75–90 minutes. **Prerequisite:** Chapter 8 and introductory logistic regression. **Exercises:** 8.1 and 8.2.

Every bundled trial and stimulus is an explicitly **synthetic teaching fixture** generated with seed 8042026. The seven WAV assets are numerical tones, not speech. The response rows are simulated and cannot provide evidence about human listeners, phoneme categories, or a population.

## Input and output contract

Inputs are a trial manifest, stimulus manifest with SHA-256 hashes, and trial-level response file. The notebook validates identifiers and assets, fits a group-descriptive logistic function and separate participant functions, derives location and slope summaries, and flags reverse or unsupported functions rather than forcing a threshold.

Outputs are `psychometric_parameters.tsv`, `psychometric_diagnostics.html`, `analysis_decisions.yaml`, `run_provenance.json`, and an unresolved `decision_record.json` under `companion/outputs/ch08/01_psychometric_functions/`.

In [ ]:
find_repository <- function(start = getwd()) {
  current <- normalizePath(start, mustWork = TRUE)
  repeat {
    if (file.exists(file.path(current, "companion", "data", "MANIFEST.tsv"))) return(current)
    parent <- dirname(current)
    if (identical(parent, current)) stop("Run from the repository root or one of its subdirectories.")
    current <- parent
  }
}
REPOSITORY_ROOT <- find_repository()
DATA_ROOT <- file.path(REPOSITORY_ROOT, "companion", "data", "ch08", "synthetic_perception")
if (!dir.exists(DATA_ROOT)) {
  stop("Chapter 8 fixture absent. Run companion/scripts/generate_ch08_fixture.py explicitly; no data are substituted silently.")
}
source(file.path(REPOSITORY_ROOT, "companion", "r", "ch08.R"))
cat("Repository:", REPOSITORY_ROOT, "
")
cat("DATA STATUS: SYNTHETIC_TEACHING_FIXTURE — no participant observations are present.
")

In [ ]:
OUTPUT <- file.path(REPOSITORY_ROOT, "companion", "outputs", "ch08", "01_psychometric_functions")
result <- run_psychometric_analysis(
  file.path(DATA_ROOT, "perception_trial_manifest.tsv"),
  file.path(DATA_ROOT, "stimulus_manifest.tsv"),
  file.path(DATA_ROOT, "psychometric_responses.tsv"),
  OUTPUT
)
stopifnot(nrow(result$parameters) == 13)
stopifnot(any(grepl("nonmonotonic_or_reverse", result$parameters$diagnostic_flag)))
print(result$parameters)

## Model and claim audit

Inspect the participant flagged as reverse or nonmonotonic. Compare the group location with participant locations, then refit after changing the sampled continuum points or excluding endpoints. Explain which changes concern the response model and which would change the theoretical claim. Do not describe a logistic shape alone as evidence of categorical perception.

In [ ]:
learner_decision <- list(
  data_status = "SYNTHETIC_TEACHING_FIXTURE",
  decision_status = "undecided",
  classification_function_claim = "Learner response required before assessment.",
  categorical_perception_claim = "not supported by this synthetic classification task",
  warning = "This record is an instructional prompt, not an empirical conclusion."
)
jsonlite::write_json(learner_decision, file.path(OUTPUT, "decision_record.json"), pretty = TRUE, auto_unbox = TRUE)
required_outputs <- c("psychometric_parameters.tsv", "psychometric_diagnostics.html", "analysis_decisions.yaml", "run_provenance.json", "decision_record.json")
stopifnot(all(file.exists(file.path(OUTPUT, required_outputs))))
print(learner_decision)